# Week 12: Atlas + Supabase Dataset Loading Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/week12_atlas_supabase_dataset_demo.ipynb)

This notebook demonstrates a beginner-friendly data pipeline:

1. pull a real dataset from an easy public JSON feed/API
2. load a local sample CSV version of the same dataset
3. clean the data into database-friendly records
4. write the records to MongoDB Atlas
5. write the records to Supabase/Postgres
6. export small local files that could be uploaded manually through Compass or Supabase

Dataset used: CISA Known Exploited Vulnerabilities (KEV) catalog.

## Safety rules before running database cells

Do not paste secrets into a notebook that will be committed to GitHub.

Use Colab Secrets or environment variables for:

- `MONGODB_URI`
- `SUPABASE_DB_URL`
- `SUPABASE_URL`
- `SUPABASE_ANON_KEY`

Never share:

- full MongoDB connection strings
- database passwords
- Supabase service role keys
- `.env` files
- screenshots that expose secrets

## Official references used in this notebook

- [CISA Known Exploited Vulnerabilities Catalog](https://www.cisa.gov/known-exploited-vulnerabilities-catalog)
- [CISA KEV data files on GitHub](https://github.com/cisagov/kev-data)
- [MongoDB PyMongo get started guide](https://www.mongodb.com/docs/languages/python/pymongo-driver/get-started/)
- [MongoDB Atlas driver connection docs](https://www.mongodb.com/docs/atlas/driver-connection/)
- [Supabase import data docs](https://supabase.com/docs/guides/database/import-data)
- [Supabase Python upsert docs](https://supabase.com/docs/reference/python/upsert)
- [PostgreSQL COPY docs](https://www.postgresql.org/docs/current/sql-copy.html)


In [ ]:
# Colab setup. Run this first in Google Colab.
# It is fine if Colab says some packages are already installed.
%pip -q install pandas requests "pymongo[srv]" sqlalchemy "psycopg[binary]" supabase python-dotenv

In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests

CISA_JSON_FEED_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
CISA_CSV_MIRROR_URL = "https://raw.githubusercontent.com/cisagov/kev-data/develop/known_exploited_vulnerabilities.csv"
COURSE_SAMPLE_CSV_URL = "https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_12/sample_cisa_kev_vulnerabilities.csv"
LOCAL_SAMPLE_CSV = Path("sample_cisa_kev_vulnerabilities.csv")

DB_NAME = "cst4714_week12"
MONGO_COLLECTION = "cisa_kev_demo"
POSTGRES_TABLE = "cisa_kev_demo"

print("Notebook ready")

## Part 1: Pull the dataset from an easy public JSON feed

The CISA KEV catalog is useful for this course because it is:

- official public-sector data
- security/admin related
- available as JSON and CSV
- small enough for beginner database practice
- structured enough to fit either MongoDB Atlas or Postgres/Supabase

In [ ]:
response = requests.get(CISA_JSON_FEED_URL, timeout=30)
response.raise_for_status()
kev_payload = response.json()

print("Catalog title:", kev_payload.get("title"))
print("Catalog version:", kev_payload.get("catalogVersion"))
print("Date released:", kev_payload.get("dateReleased"))
print("Count according to feed:", kev_payload.get("count"))

api_df = pd.DataFrame(kev_payload["vulnerabilities"])
print(api_df.shape)
api_df.head()

In [ ]:
# Save a small API dump locally so students can see what "dumping from an API" means.
api_sample_df = api_df.head(100).copy()
api_sample_df.to_csv("cisa_kev_from_api_sample.csv", index=False)
api_sample_df.to_json("cisa_kev_from_api_sample.json", orient="records", indent=2)

print("Wrote cisa_kev_from_api_sample.csv")
print("Wrote cisa_kev_from_api_sample.json")

## Part 2: Load the provided sample CSV

The course repo includes `sample_cisa_kev_vulnerabilities.csv` in the Week 12 folder.

In Colab, the notebook tries the course GitHub raw URL first. If that is unavailable, it falls back to CISA's GitHub mirror and keeps only the first 200 rows.

In [ ]:
if LOCAL_SAMPLE_CSV.exists():
    csv_df = pd.read_csv(LOCAL_SAMPLE_CSV)
    source_used = str(LOCAL_SAMPLE_CSV)
else:
    try:
        csv_df = pd.read_csv(COURSE_SAMPLE_CSV_URL)
        source_used = COURSE_SAMPLE_CSV_URL
    except Exception as exc:
        print("Course sample was not available yet; falling back to the CISA GitHub mirror.")
        print(type(exc).__name__, exc)
        csv_df = pd.read_csv(CISA_CSV_MIRROR_URL).head(200)
        source_used = CISA_CSV_MIRROR_URL

print("CSV source used:", source_used)
print(csv_df.shape)
csv_df.head()

In [ ]:
print("CSV columns")
for col in csv_df.columns:
    print("-", col)

print("\nMissing values by column")
print(csv_df.isna().sum())

## Part 3: Clean the data for database loading

The original source uses names like `cveID` and `vendorProject`.
For database practice, this notebook converts the columns to simple snake_case names.

In [ ]:
COLUMN_MAP = {
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "product": "product",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use",
    "notes": "notes",
    "cwes": "cwes",
}

KEEP_COLUMNS = list(COLUMN_MAP.values())

def clean_kev_dataframe(df: pd.DataFrame, limit: int = 100) -> pd.DataFrame:
    cleaned = df.rename(columns=COLUMN_MAP).copy()
    cleaned = cleaned[[col for col in KEEP_COLUMNS if col in cleaned.columns]]
    cleaned = cleaned.head(limit)

    for date_col in ["date_added", "due_date"]:
        if date_col in cleaned.columns:
            cleaned[date_col] = pd.to_datetime(cleaned[date_col], errors="coerce").dt.strftime("%Y-%m-%d")

    cleaned["source_name"] = "CISA Known Exploited Vulnerabilities Catalog"
    cleaned["source_url"] = CISA_JSON_FEED_URL
    cleaned["loaded_for_course"] = "CST4714 Week 12"
    cleaned["prepared_at_utc"] = datetime.now(timezone.utc).isoformat()

    # Convert NaN to None so JSON/database inserts are clean.
    cleaned = cleaned.where(pd.notna(cleaned), None)
    return cleaned

working_df = clean_kev_dataframe(csv_df, limit=100)
print(working_df.shape)
working_df.head()

In [ ]:
# These files are useful for manual upload practice.
working_df.to_csv("cisa_kev_clean_for_supabase.csv", index=False)
working_df.to_json("cisa_kev_clean_for_atlas.json", orient="records", indent=2)

records = working_df.to_dict(orient="records")

print("Prepared", len(records), "records")
print("Wrote cisa_kev_clean_for_supabase.csv")
print("Wrote cisa_kev_clean_for_atlas.json")
pprint(records[0])

## Part 4: Helper for Colab Secrets

In Colab, click the key icon on the left and add secrets with these names:

- `MONGODB_URI`
- `SUPABASE_DB_URL`
- `SUPABASE_URL`
- `SUPABASE_ANON_KEY`

If a secret is missing, the database cells will skip instead of failing immediately.

In [ ]:
def get_secret(name: str) -> str | None:
    """Read a secret from Colab Secrets or environment variables."""
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
    except Exception:
        value = None

    if not value:
        value = os.environ.get(name)

    return value

for secret_name in ["MONGODB_URI", "SUPABASE_DB_URL", "SUPABASE_URL", "SUPABASE_ANON_KEY"]:
    print(secret_name, "set" if get_secret(secret_name) else "not set")

## Part 5: Write the dataset to MongoDB Atlas

Before running this cell:

1. create a MongoDB Atlas cluster
2. create a database user
3. add your current IP address to Network Access
4. save your connection string as the Colab secret `MONGODB_URI`
5. change `RUN_ATLAS_WRITE` to `True`

This code uses an upsert so running the cell twice should not create duplicate CVE records.

In [ ]:
RUN_ATLAS_WRITE = False

if not RUN_ATLAS_WRITE:
    print("Skipping Atlas write. Set RUN_ATLAS_WRITE = True after adding MONGODB_URI as a secret.")
else:
    from pymongo import MongoClient, UpdateOne
    from pymongo.server_api import ServerApi

    mongodb_uri = get_secret("MONGODB_URI") or getpass("Paste MongoDB Atlas URI: ")

    client = MongoClient(
        mongodb_uri,
        server_api=ServerApi("1"),
        serverSelectionTimeoutMS=8000,
    )

    client.admin.command("ping")
    print("Connected to MongoDB Atlas")

    collection = client[DB_NAME][MONGO_COLLECTION]
    operations = [
        UpdateOne({"cve_id": row["cve_id"]}, {"$set": row}, upsert=True)
        for row in records
    ]

    result = collection.bulk_write(operations, ordered=False)
    print("Matched:", result.matched_count)
    print("Modified:", result.modified_count)
    print("Upserted:", len(result.upserted_ids))
    print("Collection count:", collection.count_documents({}))

    atlas_preview = list(collection.find({}, {"_id": 0}).limit(5))
    Path("atlas_roundtrip_sample.json").write_text(json.dumps(atlas_preview, indent=2), encoding="utf-8")
    print("Wrote atlas_roundtrip_sample.json")
    pd.DataFrame(atlas_preview)

### Manual MongoDB Atlas upload option

If you do not want to write from Python yet, upload `cisa_kev_clean_for_atlas.json` manually with MongoDB Compass:

1. connect Compass to Atlas
2. choose or create database `cst4714_week12`
3. create collection `cisa_kev_demo`
4. import `cisa_kev_clean_for_atlas.json`
5. inspect sample documents and create an index on `cve_id`

## Part 6: Write the dataset to Supabase/Postgres

This path uses a Supabase Postgres connection string, not the public REST API.
It is usually the cleanest demo for database administration because it shows table creation and SQL inserts.

Before running:

1. create a Supabase project
2. copy the Postgres connection string
3. save it as the Colab secret `SUPABASE_DB_URL`
4. change `RUN_SUPABASE_POSTGRES_WRITE` to `True`

In [ ]:
CREATE_TABLE_SQL = f"""
create table if not exists public.{POSTGRES_TABLE} (
    cve_id text primary key,
    vendor_project text,
    product text,
    vulnerability_name text,
    date_added date,
    short_description text,
    required_action text,
    due_date date,
    known_ransomware_campaign_use text,
    notes text,
    cwes text,
    source_name text,
    source_url text,
    loaded_for_course text,
    prepared_at_utc timestamptz
);
"""

print(CREATE_TABLE_SQL)

In [ ]:
def normalize_sqlalchemy_postgres_url(raw_url: str) -> str:
    """Force SQLAlchemy to use the psycopg v3 driver."""
    if raw_url.startswith("postgresql+psycopg://"):
        return raw_url
    if raw_url.startswith("postgresql://"):
        return "postgresql+psycopg://" + raw_url.removeprefix("postgresql://")
    if raw_url.startswith("postgres://"):
        return "postgresql+psycopg://" + raw_url.removeprefix("postgres://")
    return raw_url

UPSERT_SQL = f"""
insert into public.{POSTGRES_TABLE} (
    cve_id,
    vendor_project,
    product,
    vulnerability_name,
    date_added,
    short_description,
    required_action,
    due_date,
    known_ransomware_campaign_use,
    notes,
    cwes,
    source_name,
    source_url,
    loaded_for_course,
    prepared_at_utc
)
values (
    :cve_id,
    :vendor_project,
    :product,
    :vulnerability_name,
    :date_added,
    :short_description,
    :required_action,
    :due_date,
    :known_ransomware_campaign_use,
    :notes,
    :cwes,
    :source_name,
    :source_url,
    :loaded_for_course,
    :prepared_at_utc
)
on conflict (cve_id) do update set
    vendor_project = excluded.vendor_project,
    product = excluded.product,
    vulnerability_name = excluded.vulnerability_name,
    date_added = excluded.date_added,
    short_description = excluded.short_description,
    required_action = excluded.required_action,
    due_date = excluded.due_date,
    known_ransomware_campaign_use = excluded.known_ransomware_campaign_use,
    notes = excluded.notes,
    cwes = excluded.cwes,
    source_name = excluded.source_name,
    source_url = excluded.source_url,
    loaded_for_course = excluded.loaded_for_course,
    prepared_at_utc = excluded.prepared_at_utc;
"""

print("SQL prepared")

In [ ]:
RUN_SUPABASE_POSTGRES_WRITE = False

if not RUN_SUPABASE_POSTGRES_WRITE:
    print("Skipping Supabase/Postgres write. Set RUN_SUPABASE_POSTGRES_WRITE = True after adding SUPABASE_DB_URL as a secret.")
else:
    from sqlalchemy import create_engine, text

    raw_db_url = get_secret("SUPABASE_DB_URL") or getpass("Paste Supabase Postgres connection string: ")
    db_url = normalize_sqlalchemy_postgres_url(raw_db_url)
    engine = create_engine(db_url, pool_pre_ping=True)

    with engine.begin() as conn:
        conn.execute(text(CREATE_TABLE_SQL))
        conn.execute(text(UPSERT_SQL), records)
        count = conn.execute(text(f"select count(*) from public.{POSTGRES_TABLE}")).scalar_one()
        preview = conn.execute(text(f"select * from public.{POSTGRES_TABLE} order by date_added desc nulls last limit 5")).mappings().all()

    print("Rows in table:", count)
    supabase_preview = [dict(row) for row in preview]
    Path("supabase_roundtrip_sample.json").write_text(json.dumps(supabase_preview, indent=2, default=str), encoding="utf-8")
    print("Wrote supabase_roundtrip_sample.json")
    pd.DataFrame(supabase_preview)

### Manual Supabase upload option

If you do not want to write from Python yet, upload `cisa_kev_clean_for_supabase.csv` manually:

1. open Supabase
2. create a table named `cisa_kev_demo`
3. use the SQL table sketch above, or create matching columns in the dashboard
4. import `cisa_kev_clean_for_supabase.csv`
5. check row count and inspect records
6. add an index or primary key on `cve_id`

## Optional: Supabase Python API upsert

This option uses Supabase's HTTP API through the `supabase` Python package.
It is useful for app-style access, but the table must allow the operation through RLS/policies.

Do not put a Supabase `service_role` key in a notebook you will share.
Use a class demo project and the normal anon key only if your table policy allows the intended insert/upsert.

In [ ]:
RUN_SUPABASE_API_UPSERT = False

if not RUN_SUPABASE_API_UPSERT:
    print("Skipping Supabase API upsert. Set RUN_SUPABASE_API_UPSERT = True only after your table and policies are ready.")
else:
    from supabase import create_client

    supabase_url = get_secret("SUPABASE_URL") or getpass("Paste Supabase project URL: ")
    supabase_key = get_secret("SUPABASE_ANON_KEY") or getpass("Paste Supabase anon key: ")
    supabase = create_client(supabase_url, supabase_key)

    response = supabase.table(POSTGRES_TABLE).upsert(records, on_conflict="cve_id").execute()
    print("Upsert response:")
    pprint(response)

    preview_response = supabase.table(POSTGRES_TABLE).select("*").limit(5).execute()
    pd.DataFrame(preview_response.data)

## What students should understand after this notebook

You should be able to explain:

- why a JSON feed/API and a CSV download can represent the same dataset
- why data cleaning happens before database loading
- why Atlas naturally stores records as documents
- why Supabase/Postgres needs a table definition and data types
- why secrets must stay out of notebooks and screenshots
- why the same source dataset can be modeled differently depending on the platform